# Week 2 – Day 7
## End-to-End EDA Pipeline

This notebook demonstrates a complete EDA workflow:
- Data inspection
- Cleaning
- EDA
- Leakage detection
- ML readiness checks

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


## Create Raw Dataset (Simulating Production Data)

In [ ]:

raw_data = {
    "customer_id": [1,2,3,4,5,6,7,8,9],
    "age": [25, 30, None, 45, 200, 35, 40, None, 28],
    "category": ["A", "B", "A", "C", "B", "C", "C", "A", "B"],
    "tenure_months": [2, 5, 8, 12, 1, 36, 48, 3, 6],
    "monthly_spend": [100, 150, 120, 200, 3000, 180, 220, 130, 160],
    "total_spend": [200, 750, 960, 2400, 3000, 6480, 10560, 390, 960],
    "churn": [1, 0, 0, 0, 1, 0, 0, 1, 0]
}

df = pd.DataFrame(raw_data)
df


## Initial Inspection

In [ ]:

df.info()
df.describe()
df.isna().sum()


## Data Cleaning

In [ ]:

df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())
df.loc[df["age"] > 100, "age"] = df["age"].median()

Q1 = df["monthly_spend"].quantile(0.25)
Q3 = df["monthly_spend"].quantile(0.75)
IQR = Q3 - Q1

df["monthly_spend"] = df["monthly_spend"].clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR
)

df


## Distribution Analysis

In [ ]:

df["monthly_spend"].hist(bins=20)
plt.title("Monthly Spend Distribution")
plt.show()

sns.boxplot(x=df["monthly_spend"])
plt.title("Monthly Spend Boxplot")
plt.show()


## Categorical Analysis

In [ ]:

df["category"].value_counts(normalize=True) * 100

sns.countplot(data=df, x="category")
plt.show()


## Feature vs Target Analysis

In [ ]:

df.groupby("churn")["monthly_spend"].mean()

pd.crosstab(df["category"], df["churn"], normalize="index")


## Correlation & Leakage Check

In [ ]:

corr = df.corr(numeric_only=True)

sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.show()

corr["churn"].sort_values(ascending=False)


## Remove Leakage & Final Validation

In [ ]:

df = df.drop(columns=["total_spend"])

df.info()
df.describe()
df.isna().sum()


## Final Notes
This dataset is now cleaned, explored, and validated for ML readiness.